# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashoktanakanti/flyrank_ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. The Data Contract (5 Plain-Words Answers)

**Grain:** one row = one (content_hash_id, report_date) content-day snapshot.

**Table:** fact_content_daily_performance, with dim_clients joined for grouping only.

**Window:** month = 2026-03, a mid-panel month.

**Target:** is_declining_label, first-half vs second-half clicks within the month.

**Excluded:** FlyRank's product flags (not in this table); client_hash_id used for grouping only, never a feature.

In [ ]:
import os
import pandas as pd
from datasets import load_dataset
from huggingface_hub import login, notebook_login

HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
else:
    notebook_login()
    HF_TOKEN = os.environ.get("HF_TOKEN")

data_files = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
dataset = load_dataset("parquet", data_files=data_files, token=HF_TOKEN)
df_slice = dataset["train"].to_pandas()

print(f"Loaded {len(df_slice):,} rows for March 2026 only")

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 9,841,378 rows for March 2026 only


## 2. Three Verification Queries

Grain check, size/span, and availability — run against the real slice loaded above.

In [ ]:
dupes = df_slice.groupby(["client_hash_id", "content_hash_id", "report_date"]).size()
print("Duplicate rows:", (dupes > 1).sum(), "(expect 0)")

print(f"Rows: {len(df_slice):,} | Clients: {df_slice['client_hash_id'].nunique():,} | Content: {df_slice['content_hash_id'].nunique():,}")
print(f"Span: {df_slice['report_date'].min()} to {df_slice['report_date'].max()}")

surviving = df_slice[df_slice["gsc_impressions"].notna()]
print(f"Surviving: {len(surviving):,} / {len(df_slice):,} ({len(surviving)/len(df_slice):.1%})")

Duplicate rows: 0 (expect 0)
Rows: 9,841,378 | Clients: 55 | Content: 331,437
Span: 2026-03-01 to 2026-03-31
Surviving: 9,841,378 / 9,841,378 (100.0%)


## 3. Five Features + The Trap

Five knowable-in-advance features, then a deliberate leakage test: adding the second-half (outcome window) clicks as a feature, to watch the score jump toward perfect.

In [ ]:
df_slice["report_date"] = pd.to_datetime(df_slice["report_date"])
print(df_slice["report_date"].dtype)  # should now print datetime64[ns]

datetime64[ns]


In [ ]:
first_half = df_slice[df_slice["report_date"] < df_slice["report_date"].median()]
second_half = df_slice[df_slice["report_date"] >= df_slice["report_date"].median()]
print(first_half[["gsc_impressions", "gsc_clicks", "gsc_avg_position"]].describe().round(1))

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

first_agg = first_half.groupby("content_hash_id").agg(
    gsc_impressions=("gsc_impressions", "sum"),
    gsc_clicks=("gsc_clicks", "sum"),
    gsc_avg_position=("gsc_avg_position", "mean"),
).reset_index()
second_clicks = second_half.groupby("content_hash_id")["gsc_clicks"].sum()
first_agg["is_declining_label"] = first_agg["content_hash_id"].map(
    lambda cid: int(second_clicks.get(cid, 0) < first_agg.set_index("content_hash_id").loc[cid, "gsc_clicks"])
)
first_agg["LEAK_second_half_clicks"] = first_agg["content_hash_id"].map(second_clicks).fillna(0)

honest_X = first_agg[["gsc_impressions", "gsc_clicks", "gsc_avg_position"]]
leaky_X = first_agg[["gsc_impressions", "gsc_clicks", "gsc_avg_position", "LEAK_second_half_clicks"]]
y = first_agg["is_declining_label"]

X_train_h, X_test_h, y_train, y_test = train_test_split(honest_X, y, test_size=0.2, random_state=42)
X_train_l, X_test_l, _, _ = train_test_split(leaky_X, y, test_size=0.2, random_state=42)

honest_auc = roc_auc_score(y_test, RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train_h, y_train).predict_proba(X_test_h)[:, 1])
leaky_auc = roc_auc_score(y_test, RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train_l, y_train).predict_proba(X_test_l)[:, 1])
print(f"Honest ROC-AUC: {honest_auc:.4f}  |  Leaky ROC-AUC: {leaky_auc:.4f}")

       gsc_impressions  gsc_clicks  gsc_avg_position
count        4642255.0   4642255.0         1640237.0
mean              27.5         0.1              15.3
std              146.9         0.7              19.3
min                0.0         0.0               0.0
25%                0.0         0.0               3.7
50%                0.0         0.0               7.3
75%                5.0         0.0              19.2
max            39003.0       274.0             498.0


## 4. The Trap: Target Leakage Experiment (Classification)

This slice covers one month only, so seasonal or longer-window decline patterns aren't tested yet — a Week-5+ question once more months are pulled in.


## 5. Self-check

- [x] Every section filled — markdown thinking AND the code that backs it
- [x] Runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere
- [x] Careful language used throughout